In [20]:
# Project Variables

lakehouse_name = "lh_dev_ecommerce"
env_name = "dev"
job_id = 102

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 24, Finished, Available, Finished, False)

In [21]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_stage_db
""")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 25, Finished, Available, Finished, False)

DataFrame[]

In [22]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {lakehouse_name}.{env_name}_reject_db
""")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 26, Finished, Available, Finished, False)

DataFrame[]

In [23]:
metadata_df = spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_metadata_db.landing_metadata_tbl
WHERE JobID = 101
""")

metadata_list = metadata_df.collect()

metadata_df.show(truncate=False)

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 27, Finished, Available, Finished, False)

+-----+---------------+---------------+---------------+
|JobID|LandingFilePath|LandingFileName|BronzeTableName|
+-----+---------------+---------------+---------------+
|101  |Files/landing/ |order_items.csv|order_items    |
|101  |Files/landing/ |customers.csv  |customers      |
|101  |Files/landing/ |products.csv   |products       |
|101  |Files/landing/ |orders.csv     |orders         |
+-----+---------------+---------------+---------------+



In [24]:
spark.sql(f"""
CREATE OR REPLACE TABLE {lakehouse_name}.{env_name}_metadata_db.validation_rules_tbl
(
    TableName STRING,
    ColumnName STRING,
    ValidationRule STRING,
    ErrorMessage STRING
)
""")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 28, Finished, Available, Finished, False)

DataFrame[]

In [25]:
validation_df = spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_metadata_db.validation_rules_tbl
""")

validation_df.show()

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 29, Finished, Available, Finished, False)

+---------+----------+--------------+------------+
|TableName|ColumnName|ValidationRule|ErrorMessage|
+---------+----------+--------------+------------+
+---------+----------+--------------+------------+



In [26]:
spark.sql(f"""
INSERT INTO {lakehouse_name}.{env_name}_metadata_db.validation_rules_tbl
VALUES

('customers','CustomerID','NOT_NULL','CustomerID cannot be NULL'),
('customers','CustomerName','NOT_NULL','CustomerName cannot be NULL'),

('orders','OrderID','NOT_NULL','OrderID cannot be NULL'),
('orders','CustomerID','NOT_NULL','CustomerID cannot be NULL'),

('products','ProductID','NOT_NULL','ProductID cannot be NULL'),
('products','ProductName','NOT_NULL','ProductName cannot be NULL'),

('order_items','OrderItemID','NOT_NULL','OrderItemID cannot be NULL'),
('order_items','OrderID','NOT_NULL','OrderID cannot be NULL'),
('order_items','ProductID','NOT_NULL','ProductID cannot be NULL')
""")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 30, Finished, Available, Finished, False)

DataFrame[]

In [27]:
validation_df = spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_metadata_db.validation_rules_tbl
""")

validation_df.show(truncate=False)

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 31, Finished, Available, Finished, False)

+-----------+------------+--------------+---------------------------+
|TableName  |ColumnName  |ValidationRule|ErrorMessage               |
+-----------+------------+--------------+---------------------------+
|customers  |CustomerName|NOT_NULL      |CustomerName cannot be NULL|
|order_items|OrderItemID |NOT_NULL      |OrderItemID cannot be NULL |
|products   |ProductName |NOT_NULL      |ProductName cannot be NULL |
|order_items|OrderID     |NOT_NULL      |OrderID cannot be NULL     |
|order_items|ProductID   |NOT_NULL      |ProductID cannot be NULL   |
|customers  |CustomerID  |NOT_NULL      |CustomerID cannot be NULL  |
|products   |ProductID   |NOT_NULL      |ProductID cannot be NULL   |
|orders     |CustomerID  |NOT_NULL      |CustomerID cannot be NULL  |
|orders     |OrderID     |NOT_NULL      |OrderID cannot be NULL     |
+-----------+------------+--------------+---------------------------+



In [28]:
rules = {}

for row in validation_df.collect():

    table = row["TableName"]

    if table not in rules:
        rules[table] = []

    rules[table].append({
        "Column": row["ColumnName"],
        "Rule": row["ValidationRule"],
        "Message": row["ErrorMessage"]
    })

print(rules)

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 32, Finished, Available, Finished, False)

{'customers': [{'Column': 'CustomerName', 'Rule': 'NOT_NULL', 'Message': 'CustomerName cannot be NULL'}, {'Column': 'CustomerID', 'Rule': 'NOT_NULL', 'Message': 'CustomerID cannot be NULL'}], 'order_items': [{'Column': 'OrderItemID', 'Rule': 'NOT_NULL', 'Message': 'OrderItemID cannot be NULL'}, {'Column': 'OrderID', 'Rule': 'NOT_NULL', 'Message': 'OrderID cannot be NULL'}, {'Column': 'ProductID', 'Rule': 'NOT_NULL', 'Message': 'ProductID cannot be NULL'}], 'products': [{'Column': 'ProductName', 'Rule': 'NOT_NULL', 'Message': 'ProductName cannot be NULL'}, {'Column': 'ProductID', 'Rule': 'NOT_NULL', 'Message': 'ProductID cannot be NULL'}], 'orders': [{'Column': 'CustomerID', 'Rule': 'NOT_NULL', 'Message': 'CustomerID cannot be NULL'}, {'Column': 'OrderID', 'Rule': 'NOT_NULL', 'Message': 'OrderID cannot be NULL'}]}


In [29]:
from pyspark.sql.functions import col
from functools import reduce

valid_dfs = {}

for table_name in rules.keys():

    print("=" * 60)
    print(f"Processing : {table_name}")

    bronze_df = spark.table(
        f"{lakehouse_name}.{env_name}_bronze_db.{table_name}"
    )

    conditions = []

    for rule in rules[table_name]:

        column_name = rule["Column"]

        if column_name in bronze_df.columns:

            if rule["Rule"] == "NOT_NULL":
                conditions.append(col(column_name).isNotNull())

        else:
            print(f"Warning: Column '{column_name}' not found in {table_name}")

    if conditions:
        valid_df = bronze_df.filter(reduce(lambda x, y: x & y, conditions))
    else:
        valid_df = bronze_df

    valid_dfs[table_name] = valid_df

    print(f"Valid Records : {valid_df.count()}")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 33, Finished, Available, Finished, False)

Processing : customers
Valid Records : 26
Processing : order_items
Valid Records : 37
Processing : products
Valid Records : 25
Processing : orders
Valid Records : 25


In [30]:
from pyspark.sql.functions import col
from functools import reduce

reject_dfs = {}

for table_name in rules.keys():

    bronze_df = spark.table(
        f"{lakehouse_name}.{env_name}_bronze_db.{table_name}"
    )

    reject_conditions = []

    for rule in rules[table_name]:

        column_name = rule["Column"]

        if column_name in bronze_df.columns:

            if rule["Rule"] == "NOT_NULL":
                reject_conditions.append(col(column_name).isNull())

    if reject_conditions:
        reject_df = bronze_df.filter(
            reduce(lambda x, y: x | y, reject_conditions)
        )
    else:
        reject_df = spark.createDataFrame([], bronze_df.schema)

    reject_dfs[table_name] = reject_df

    print(f"Reject Records : {reject_df.count()}")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 34, Finished, Available, Finished, False)

Reject Records : 0
Reject Records : 0
Reject Records : 0
Reject Records : 0


In [31]:
for table_name, valid_df in valid_dfs.items():

    valid_df.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(
            f"{lakehouse_name}.{env_name}_stage_db.{table_name}"
        )

    print(f"✅ {table_name} loaded to Stage")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 35, Finished, Available, Finished, False)

✅ customers loaded to Stage
✅ order_items loaded to Stage
✅ products loaded to Stage
✅ orders loaded to Stage


In [32]:
from pyspark.sql.functions import when, lit, col

reject_dfs = {}

for table_name in rules.keys():

    bronze_df = spark.table(
        f"{lakehouse_name}.{env_name}_bronze_db.{table_name}"
    )

    reject_df = bronze_df

    reject_reason = None

    for rule in rules[table_name]:

        column_name = rule["Column"]

        if column_name in bronze_df.columns:

            condition = col(column_name).isNull()

            if reject_reason is None:
                reject_reason = when(condition, lit(rule["Message"]))
            else:
                reject_reason = reject_reason.when(condition, lit(rule["Message"]))

    if reject_reason is not None:
        reject_df = reject_df.withColumn("RejectReason", reject_reason)
        reject_df = reject_df.filter(col("RejectReason").isNotNull())

    reject_dfs[table_name] = reject_df

    print(f"{table_name} Reject Records : {reject_df.count()}")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 36, Finished, Available, Finished, False)

customers Reject Records : 0
order_items Reject Records : 0
products Reject Records : 0
orders Reject Records : 0


In [33]:
for table_name, reject_df in reject_dfs.items():

    reject_df.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(
            f"{lakehouse_name}.{env_name}_reject_db.{table_name}_reject"
        )

    print(f"✅ {table_name} Reject Table Created")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 37, Finished, Available, Finished, False)

✅ customers Reject Table Created
✅ order_items Reject Table Created
✅ products Reject Table Created
✅ orders Reject Table Created


In [34]:
for table_name in rules.keys():

    bronze_count = spark.table(
        f"{lakehouse_name}.{env_name}_bronze_db.{table_name}"
    ).count()

    stage_count = spark.table(
        f"{lakehouse_name}.{env_name}_stage_db.{table_name}"
    ).count()

    reject_count = spark.table(
        f"{lakehouse_name}.{env_name}_reject_db.{table_name}_reject"
    ).count()

    print("=" * 60)
    print(f"Table : {table_name}")
    print(f"Bronze Records : {bronze_count}")
    print(f"Stage Records  : {stage_count}")
    print(f"Reject Records : {reject_count}")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 38, Finished, Available, Finished, False)

Table : customers
Bronze Records : 26
Stage Records  : 26
Reject Records : 0
Table : order_items
Bronze Records : 37
Stage Records  : 37
Reject Records : 0
Table : products
Bronze Records : 25
Stage Records  : 25
Reject Records : 0
Table : orders
Bronze Records : 25
Stage Records  : 25
Reject Records : 0


In [35]:
from datetime import datetime
from pyspark.sql.types import *

audit_rows = []

for table_name in rules.keys():

    stage_count = spark.table(
        f"{lakehouse_name}.{env_name}_stage_db.{table_name}"
    ).count()

    audit_rows.append(
        (
            102,
            "Validation_Bronze_to_Stage",
            table_name,
            "Success",
            stage_count,
            datetime.now(),
            datetime.now(),
            "Validation Completed"
        )
    )

schema = StructType([
    StructField("JobID", IntegerType(), True),
    StructField("JobName", StringType(), True),
    StructField("TableName", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("RowsProcessed", IntegerType(), True),
    StructField("StartTime", TimestampType(), True),
    StructField("EndTime", TimestampType(), True),
    StructField("Message", StringType(), True)
])

audit_df = spark.createDataFrame(audit_rows, schema)

audit_df.write \
.mode("append") \
.saveAsTable(
    f"{lakehouse_name}.{env_name}_audit_db.job_event_log_tbl"
)

print("✅ Audit Updated Successfully")

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 39, Finished, Available, Finished, False)

✅ Audit Updated Successfully


In [36]:
spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_audit_db.job_event_log_tbl
ORDER BY StartTime DESC
""").show(truncate=False)

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 40, Finished, Available, Finished, False)

+-----+--------------------------+-----------+-------+-------------+--------------------------+--------------------------+---------------------+
|JobID|JobName                   |TableName  |Status |RowsProcessed|StartTime                 |EndTime                   |Message              |
+-----+--------------------------+-----------+-------+-------------+--------------------------+--------------------------+---------------------+
|102  |Validation_Bronze_to_Stage|orders     |Success|25           |2026-07-10 11:30:40.605462|2026-07-10 11:30:40.605465|Validation Completed |
|102  |Validation_Bronze_to_Stage|products   |Success|25           |2026-07-10 11:30:40.10967 |2026-07-10 11:30:40.109674|Validation Completed |
|102  |Validation_Bronze_to_Stage|order_items|Success|37           |2026-07-10 11:30:39.557352|2026-07-10 11:30:39.557355|Validation Completed |
|102  |Validation_Bronze_to_Stage|customers  |Success|26           |2026-07-10 11:30:39.078021|2026-07-10 11:30:39.078025|Validati

In [37]:
spark.sql(f"""
SELECT OrderID, OrderDate
FROM {lakehouse_name}.{env_name}_stage_db.orders
LIMIT 10
""").show(truncate=False)

StatementMeta(, 57f1afc6-c933-4898-8d58-b197937a6a0c, 42, Finished, Available, Finished, False)

+-------+----------+
|OrderID|OrderDate |
+-------+----------+
|ORD001 |2023/02/04|
|ORD002 |2023/10/20|
|ORD003 |2023/08/05|
|ORD004 |2023/05/08|
|ORD005 |2023/02/20|
|ORD006 |2023/03/26|
|ORD007 |2023/04/08|
|ORD008 |2023/09/15|
|ORD009 |2023/03/03|
|ORD010 |2023/05/14|
+-------+----------+

